In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from matplotlib import pyplot as plt
from keras.datasets import fashion_mnist
from keras.models import Sequential

## Завдання 1

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape([-1, 28, 28, 1])
x_test = x_test.reshape([-1, 28, 28, 1])

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model = Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D((2, 2)),

    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(128, activation='relu'),

    keras.layers.Dense(10, activation='softmax')
])

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.00001
)

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.fit(x_train, y_train, epochs=50, batch_size=64, validation_data=(x_test, y_test), callbacks=[reduce_lr])

Epoch 1/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 13s 8ms/step - accuracy: 0.8224 - loss: 0.4893 - val_accuracy: 0.8610 - val_loss: 0.3704 - learning_rate: 0.0010
Epoch 2/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8804 - loss: 0.3292 - val_accuracy: 0.8893 - val_loss: 0.2983 - learning_rate: 0.0010
Epoch 3/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8931 - loss: 0.2909 - val_accuracy: 0.8986 - val_loss: 0.2735 - learning_rate: 0.0010
Epoch 4/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9022 - loss: 0.2636 - val_accuracy: 0.9027 - val_loss: 0.2697 - learning_rate: 0.0010
Epoch 5/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9085 - loss: 0.2466 - val_accuracy: 0.9053 - val_loss: 0.2557 - learning_rate: 0.0010
Epoch 6/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9141 - loss: 0.2337 - val_accuracy: 0.9004 - val_loss: 0.2762 - learning_rate: 0.0010
Epoch 7/50
938/938 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9182 - loss: 0.2210 -

# Висновок:

У ході виконання завдання була побудована згорткова нейронна мережа для класифікації товарів з датасету fashion_mnist. В процесі пошуку оптимальної архітектури були перевірені різні комбінації гіперпараметрів: кількість фільтрів на шарах Conv2D (на останньому шарі випробував 64 та 128) та кількість нейронів у Dense шарі (64, 128 та 256). Серед оптимізаторів були перевірені RMSprop та Adam. Adam показав кращий результат у комбінації з callback ReduceLROnPlateau, який автоматично зменшував learning rate коли val_loss переставав покращуватись — без нього val_loss та val_accuracy нестабільно коливались. Також була використана регуляризація у вигляді BatchNormalization та Dropout(0.5), що допомогло зменшити overfitting.
Фінальна архітектура: три блоки Conv2D (32→64→128 фільтрів) з BatchNormalization та MaxPooling після кожного, Dropout(0.5) після Flatten, Dense(128) та вихідний шар Dense(10, softmax). Оптимізатор — Adam з ReduceLROnPlateau.
Повнозв'язна мережа з попереднього ДЗ показала максимальний результат 90.73%, тоді як CNN досягла 93.01% на епосі 26 — приріст 2.28%. Це підтверджує що згорткові мережі краще підходять для задач класифікації зображень завдяки здатності вловлювати просторові патерни — форму, краї та текстури об'єктів.


## Завдання 2

In [ ]:
from keras.applications.vgg16 import preprocess_input, VGG16

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

x_train = tf.image.grayscale_to_rgb(tf.expand_dims(x_train, -1))
x_test = tf.image.grayscale_to_rgb(tf.expand_dims(x_test, -1))

x_train = tf.image.resize(x_train, [72, 72])
x_test = tf.image.resize(x_test, [72, 72])

x_train = preprocess_input(x_train)
x_test = preprocess_input(x_test)

In [ ]:
conv_base = VGG16(weights="imagenet", include_top=False, input_shape=(72, 72, 3))
conv_base.trainable = False

model = Sequential([
    conv_base,
    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(x_test, y_test)
)

Epoch 1/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 85s 82ms/step - accuracy: 0.8001 - loss: 0.7759 - val_accuracy: 0.8648 - val_loss: 0.3805
Epoch 2/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 68s 73ms/step - accuracy: 0.8511 - loss: 0.4284 - val_accuracy: 0.8722 - val_loss: 0.3562
Epoch 3/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 71ms/step - accuracy: 0.8600 - loss: 0.3946 - val_accuracy: 0.8757 - val_loss: 0.3460
Epoch 4/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 71ms/step - accuracy: 0.8693 - loss: 0.3692 - val_accuracy: 0.8817 - val_loss: 0.3411
Epoch 5/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 71ms/step - accuracy: 0.8730 - loss: 0.3590 - val_accuracy: 0.8843 - val_loss: 0.3484
Epoch 6/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 72ms/step - accuracy: 0.8765 - loss: 0.3463 - val_accuracy: 0.8809 - val_loss: 0.3485
Epoch 7/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 72ms/step - accuracy: 0.8812 - loss: 0.3315 - val_accuracy: 0.8864 - val_loss: 0.3336
Epoch 8/10
938/938 ━━━━━━━━━━━━━━━━━━━━ 67s 72ms/step - accuracy: 0.8835 - loss: 0.3219 - 

In [ ]:
conv_base.trainable = True
for layer in conv_base.layers:
    if "block5" in layer.name:
        layer.trainable = True
    else:
        layer.trainable = False

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5), # 0.00001
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
)

In [ ]:
history_fine = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[reduce_lr]
)

Epoch 1/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 92s 93ms/step - accuracy: 0.9028 - loss: 0.2671 - val_accuracy: 0.9026 - val_loss: 0.2914 - learning_rate: 1.0000e-05
Epoch 2/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 84s 89ms/step - accuracy: 0.9207 - loss: 0.2160 - val_accuracy: 0.9086 - val_loss: 0.2794 - learning_rate: 1.0000e-05
Epoch 3/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 84s 90ms/step - accuracy: 0.9298 - loss: 0.1853 - val_accuracy: 0.9108 - val_loss: 0.2995 - learning_rate: 1.0000e-05
Epoch 4/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 83s 89ms/step - accuracy: 0.9374 - loss: 0.1632 - val_accuracy: 0.9159 - val_loss: 0.2810 - learning_rate: 1.0000e-05
Epoch 5/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 83s 89ms/step - accuracy: 0.9454 - loss: 0.1420 - val_accuracy: 0.9191 - val_loss: 0.2988 - learning_rate: 1.0000e-05
Epoch 6/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 82s 88ms/step - accuracy: 0.9549 - loss: 0.1165 - val_accuracy: 0.9206 - val_loss: 0.3013 - learning_rate: 5.0000e-06
Epoch 7/15
938/938 ━━━━━━━━━━━━━━━━━━━━ 83s 89ms/ste

## Висновок:

Використання VGG16 для такої простої задачі, як класифікація Fashion MNIST, наочно демонструє, що потужніша архітектура не завжди гарантує кращий результат, адже моя кастомна CNN (93%) виявилася ефективнішою за важку VGG16 (92.6%). Головна проблема полягала в тому, що VGG16 розроблялася під великі кольорові фотографії, і для її роботи нам довелося штучно розтягувати маленькі зображення 28х28 до 72х72, що неминуче призвело до появи розмитості та втрати дрібних деталей. Крім того, на етапі виділення ознак (Feature Extraction) модель показала посередні результати, і лише завдяки тонкому донавчанню (Fine-tuning) останнього згорткового блоку з мікроскопічним кроком навчання нам вдалося наблизитися до показників простої мережі. У підсумку, хоча Transfer Learning і є незамінним інструментом для складних фотореалістичних об'єктів, для схематичних чорно-білих даних він виявився занадто громіздким, повільним у навчанні та ресурсомістким, підтверджуючи правило, що архітектура має відповідати складності конкретного завдання.